# Sprint 4 — Webinar 14 (Práctico) — Student Version

**Modalidad:** Coding Together + Breakout Rooms  
**Tema:** Data Journey → Funnels → Cohorts — Ejercicios aplicados en SQL  
**Motor SQL sugerido:** [SQL Workbench](https://sql-workbench.com/) con DuckDB

Este cuaderno está diseñado para que vayas construyendo tus respuestas durante la sesión.  
La idea no es solo que la query funcione, sino que puedas dejar evidencia de:

- qué entendiste del problema,
- qué consulta intentaste,
- qué resultado viste,
- y cómo lo interpretarías para negocio.

### Antes de empezar
Completa:

- **Fecha:** ______________________
- **Nombre:** ______________________
- **Compañero/a de breakout room:** ______________________

### Meta personal de la sesión
¿Qué te gustaría lograr hoy?

________________________________________________________________

________________________________________________________________

## Objetivos de aprendizaje

Marca al final de la sesión lo que lograste.

| Objetivo | ¿Lo logré? | Mis notas |
|---|---|---|
| Explorar y validar un dataset de eventos | ☐ | |
| Escribir CTEs simples y encadenadas | ☐ | |
| Construir un funnel completo con métricas de conversión | ☐ | |
| Calcular tiempos entre etapas del journey | ☐ | |
| Construir cohorts y tablas de retención | ☐ | |
| Interpretar hallazgos y proponer acciones de negocio | ☐ | |

## Estructura y timeboxing

| Tiempo | Bloque | Modalidad | Qué haré |
|-------:|--------|-----------|-----------|
| 0–10 | Bloque 0 | Breakout Rooms | Preparar entorno y validar tablas |
| 10–35 | Bloque 1 | Coding Together | Practicar CTEs paso a paso |
| 35–60 | Bloque 2 | Coding Together | Construir el funnel de conversión |
| 60–80 | Bloque 3 | Coding Together | Cohorts y retención |
| 80–95 | Bloque 4 | Breakout Rooms | Ejercicio integrador |
| 95–100 | Cierre | Plenaria | Hallazgos, takeaways y próximos pasos |

### Bitácora de clase
Anota ideas, errores o tips del docente:

- ________________________________________________________________
- ________________________________________________________________
- ________________________________________________________________

## Escenario: MusicFlow — Plataforma de Streaming

**MusicFlow** es una plataforma de streaming de música. El equipo quiere entender:

- cómo es el **journey** de un usuario nuevo,
- dónde están los **cuellos de botella** del funnel,
- qué tan bien **retienen** las cohortes,
- y si existen diferencias por **canal**, **plan**, **país** o **device**.

### Journey del producto

| Etapa | Evento | ¿Qué significa para el negocio? |
|---|---|---|
| 1 | `app_open` | |
| 2 | `search_song` | |
| 3 | `play_song` | |
| 4 | `create_playlist` | |
| 5 | `subscribe_premium` | |

### Preguntas de entrada
1. ¿Qué etapa crees que tendrá más abandono?  
   ________________________________________________________________

2. ¿Qué canal crees que traerá usuarios de mejor calidad?  
   ________________________________________________________________

3. ¿Qué revisarías primero antes de construir un funnel?  
   ________________________________________________________________

---
# Bloque 0 · Preparación del entorno

## 0.1 Checklist de preparación

Marca lo que vayas completando:

- ☐ Abrí `https://sql-workbench.com/`
- ☐ Seleccioné **DuckDB**
- ☐ Entiendo que trabajaré con dos tablas: `users` y `events`
- ☐ Tengo claro que hoy sí usaré **CTEs**
- ☐ Voy a registrar no solo resultados, también interpretación

### Antes de ejecutar
¿Qué esperas que tenga cada tabla?

| Tabla | ¿Qué tipo de información espero ver? |
|---|---|
| `users` | |
| `events` | |

## 0.2 Creación de tablas (DDL)

In [ ]:
-- =============================================
-- DDL: MusicFlow — Plataforma de Streaming
-- Compatible con DuckDB (SQL Workbench)
-- =============================================

CREATE OR REPLACE TABLE users (
    user_id     INTEGER PRIMARY KEY,
    signup_ts   TIMESTAMP NOT NULL,
    plan        VARCHAR NOT NULL,       -- 'free' o 'premium'
    channel     VARCHAR NOT NULL,       -- 'organic', 'ads', 'referral', 'social'
    device      VARCHAR NOT NULL,       -- 'mobile', 'desktop', 'tablet'
    country     VARCHAR NOT NULL        -- 'CO', 'MX', 'AR', 'PE', 'CL'
);

CREATE OR REPLACE TABLE events (
    event_id    INTEGER PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_name  VARCHAR NOT NULL,
    event_ts    TIMESTAMP NOT NULL,
    props       VARCHAR NOT NULL DEFAULT '{}'
);

## 0.3 Carga de datos (Seed)

In [ ]:
-- =============================================
-- SEED: Usuarios de MusicFlow (Q1 2024)
-- 20 usuarios distribuidos en 3 cohortes mensuales
-- =============================================

INSERT INTO users (user_id, signup_ts, plan, channel, device, country) VALUES
    -- Cohorte Enero 2024 (7 usuarios)
    (1,  '2024-01-03 08:30:00', 'free',    'organic',  'mobile',  'CO'),
    (2,  '2024-01-05 14:15:00', 'free',    'ads',      'desktop', 'MX'),
    (3,  '2024-01-08 10:00:00', 'free',    'referral', 'mobile',  'AR'),
    (4,  '2024-01-12 19:20:00', 'free',    'social',   'mobile',  'CO'),
    (5,  '2024-01-15 11:45:00', 'free',    'organic',  'tablet',  'PE'),
    (6,  '2024-01-20 16:30:00', 'free',    'ads',      'mobile',  'MX'),
    (7,  '2024-01-25 09:00:00', 'free',    'social',   'desktop', 'CL'),

    -- Cohorte Febrero 2024 (7 usuarios)
    (8,  '2024-02-02 07:45:00', 'free',    'organic',  'mobile',  'CO'),
    (9,  '2024-02-06 13:30:00', 'free',    'ads',      'mobile',  'MX'),
    (10, '2024-02-10 18:00:00', 'free',    'referral', 'desktop', 'AR'),
    (11, '2024-02-14 22:10:00', 'free',    'social',   'mobile',  'PE'),
    (12, '2024-02-18 10:15:00', 'free',    'organic',  'tablet',  'CO'),
    (13, '2024-02-22 15:00:00', 'free',    'ads',      'mobile',  'CL'),
    (14, '2024-02-26 08:30:00', 'free',    'referral', 'mobile',  'MX'),

    -- Cohorte Marzo 2024 (6 usuarios)
    (15, '2024-03-01 09:00:00', 'free',    'organic',  'mobile',  'CO'),
    (16, '2024-03-05 12:20:00', 'free',    'social',   'desktop', 'AR'),
    (17, '2024-03-10 17:45:00', 'free',    'ads',      'mobile',  'MX'),
    (18, '2024-03-14 20:30:00', 'free',    'referral', 'mobile',  'PE'),
    (19, '2024-03-20 11:00:00', 'free',    'organic',  'tablet',  'CL'),
    (20, '2024-03-25 14:15:00', 'free',    'ads',      'desktop', 'CO');

-- =============================================
-- SEED: Eventos de MusicFlow (Q1 2024)
-- Journey: app_open → search_song → play_song → create_playlist → subscribe_premium
-- =============================================

-- === ENERO: Usuarios 1-7 ===
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
    -- Usuario 1: Journey completo (convierte a premium)
    (1,  1, 'app_open',           '2024-01-03 08:35:00', '{"source":"organic"}'),
    (2,  1, 'search_song',        '2024-01-03 08:40:00', '{"query":"Bad Bunny"}'),
    (3,  1, 'play_song',          '2024-01-03 08:42:00', '{"song":"Monaco","artist":"Bad Bunny"}'),
    (4,  1, 'play_song',          '2024-01-03 08:46:00', '{"song":"Titi Me Pregunto","artist":"Bad Bunny"}'),
    (5,  1, 'create_playlist',    '2024-01-03 09:00:00', '{"name":"Reggaeton Mix"}'),
    (6,  1, 'subscribe_premium',  '2024-01-03 09:15:00', '{"plan":"monthly","price":4.99}'),

    -- Usuario 2: Llega hasta play_song, no crea playlist
    (7,  2, 'app_open',           '2024-01-05 14:20:00', '{"source":"ads"}'),
    (8,  2, 'search_song',        '2024-01-05 14:25:00', '{"query":"Shakira"}'),
    (9,  2, 'play_song',          '2024-01-05 14:28:00', '{"song":"Bzrp Session 53","artist":"Shakira"}'),

    -- Usuario 3: Journey completo (convierte)
    (10, 3, 'app_open',           '2024-01-08 10:05:00', '{"source":"referral"}'),
    (11, 3, 'search_song',        '2024-01-08 10:10:00', '{"query":"Peso Pluma"}'),
    (12, 3, 'play_song',          '2024-01-08 10:12:00', '{"song":"Ella Baila Sola","artist":"Peso Pluma"}'),
    (13, 3, 'create_playlist',    '2024-01-08 10:30:00', '{"name":"Regional Mexicano"}'),
    (14, 3, 'subscribe_premium',  '2024-01-08 10:45:00', '{"plan":"annual","price":49.99}'),

    -- Usuario 4: Solo abre la app y busca, no reproduce
    (15, 4, 'app_open',           '2024-01-12 19:25:00', '{"source":"social"}'),
    (16, 4, 'search_song',        '2024-01-12 19:30:00', '{"query":"Karol G"}'),

    -- Usuario 5: Solo abre la app (bounce)
    (17, 5, 'app_open',           '2024-01-15 11:50:00', '{"source":"organic"}'),

    -- Usuario 6: Llega hasta create_playlist pero no suscribe
    (18, 6, 'app_open',           '2024-01-20 16:35:00', '{"source":"ads"}'),
    (19, 6, 'search_song',        '2024-01-20 16:38:00', '{"query":"Feid"}'),
    (20, 6, 'play_song',          '2024-01-20 16:40:00', '{"song":"Normal","artist":"Feid"}'),
    (21, 6, 'play_song',          '2024-01-20 16:44:00', '{"song":"Ferxxo 100","artist":"Feid"}'),
    (22, 6, 'create_playlist',    '2024-01-20 17:00:00', '{"name":"Feid Hits"}'),

    -- Usuario 7: Llega hasta play_song
    (23, 7, 'app_open',           '2024-01-25 09:05:00', '{"source":"social"}'),
    (24, 7, 'search_song',        '2024-01-25 09:08:00', '{"query":"Rauw Alejandro"}'),
    (25, 7, 'play_song',          '2024-01-25 09:10:00', '{"song":"Todo De Ti","artist":"Rauw Alejandro"}');

-- === FEBRERO: Usuarios 8-14 ===
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
    -- Usuario 8: Journey completo (convierte)
    (26, 8, 'app_open',           '2024-02-02 07:50:00', '{"source":"organic"}'),
    (27, 8, 'search_song',        '2024-02-02 07:55:00', '{"query":"J Balvin"}'),
    (28, 8, 'play_song',          '2024-02-02 07:58:00', '{"song":"Mi Gente","artist":"J Balvin"}'),
    (29, 8, 'create_playlist',    '2024-02-02 08:20:00', '{"name":"Perreo"}'),
    (30, 8, 'subscribe_premium',  '2024-02-02 08:30:00', '{"plan":"monthly","price":4.99}'),

    -- Usuario 9: Solo abre y busca
    (31, 9, 'app_open',           '2024-02-06 13:35:00', '{"source":"ads"}'),
    (32, 9, 'search_song',        '2024-02-06 13:40:00', '{"query":"Maluma"}'),

    -- Usuario 10: Llega hasta play_song
    (33, 10, 'app_open',          '2024-02-10 18:05:00', '{"source":"referral"}'),
    (34, 10, 'search_song',       '2024-02-10 18:08:00', '{"query":"Ozuna"}'),
    (35, 10, 'play_song',         '2024-02-10 18:10:00', '{"song":"Se Preparó","artist":"Ozuna"}'),

    -- Usuario 11: Solo abre app (bounce)
    (36, 11, 'app_open',          '2024-02-14 22:15:00', '{"source":"social"}'),

    -- Usuario 12: Llega hasta create_playlist (no suscribe)
    (37, 12, 'app_open',          '2024-02-18 10:20:00', '{"source":"organic"}'),
    (38, 12, 'search_song',       '2024-02-18 10:25:00', '{"query":"Carlos Vives"}'),
    (39, 12, 'play_song',         '2024-02-18 10:28:00', '{"song":"La Bicicleta","artist":"Carlos Vives"}'),
    (40, 12, 'create_playlist',   '2024-02-18 10:50:00', '{"name":"Vallenato Pop"}'),

    -- Usuario 13: Journey completo (convierte)
    (41, 13, 'app_open',          '2024-02-22 15:05:00', '{"source":"ads"}'),
    (42, 13, 'search_song',       '2024-02-22 15:08:00', '{"query":"Anuel AA"}'),
    (43, 13, 'play_song',         '2024-02-22 15:10:00', '{"song":"China","artist":"Anuel AA"}'),
    (44, 13, 'create_playlist',   '2024-02-22 15:30:00', '{"name":"Trap Latino"}'),
    (45, 13, 'subscribe_premium', '2024-02-22 15:45:00', '{"plan":"monthly","price":4.99}'),

    -- Usuario 14: Llega hasta play_song
    (46, 14, 'app_open',          '2024-02-26 08:35:00', '{"source":"referral"}'),
    (47, 14, 'search_song',       '2024-02-26 08:38:00', '{"query":"Natti Natasha"}'),
    (48, 14, 'play_song',         '2024-02-26 08:40:00', '{"song":"Sin Pijama","artist":"Natti Natasha"}');

-- === MARZO: Usuarios 15-20 ===
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
    -- Usuario 15: Journey completo (convierte)
    (49, 15, 'app_open',          '2024-03-01 09:05:00', '{"source":"organic"}'),
    (50, 15, 'search_song',       '2024-03-01 09:10:00', '{"query":"Mora"}'),
    (51, 15, 'play_song',         '2024-03-01 09:12:00', '{"song":"Una Vez","artist":"Mora"}'),
    (52, 15, 'create_playlist',   '2024-03-01 09:30:00', '{"name":"Chill Reggaeton"}'),
    (53, 15, 'subscribe_premium', '2024-03-01 09:50:00', '{"plan":"annual","price":49.99}'),

    -- Usuario 16: Solo abre y busca
    (54, 16, 'app_open',          '2024-03-05 12:25:00', '{"source":"social"}'),
    (55, 16, 'search_song',       '2024-03-05 12:30:00', '{"query":"Duki"}'),

    -- Usuario 17: Llega hasta play_song
    (56, 17, 'app_open',          '2024-03-10 17:50:00', '{"source":"ads"}'),
    (57, 17, 'search_song',       '2024-03-10 17:53:00', '{"query":"Myke Towers"}'),
    (58, 17, 'play_song',         '2024-03-10 17:55:00', '{"song":"La Nota","artist":"Myke Towers"}'),

    -- Usuario 18: Llega hasta create_playlist (no suscribe)
    (59, 18, 'app_open',          '2024-03-14 20:35:00', '{"source":"referral"}'),
    (60, 18, 'search_song',       '2024-03-14 20:38:00', '{"query":"Sebastian Yatra"}'),
    (61, 18, 'play_song',         '2024-03-14 20:40:00', '{"song":"Traicionera","artist":"Sebastian Yatra"}'),
    (62, 18, 'create_playlist',   '2024-03-14 21:00:00', '{"name":"Pop Latino"}'),

    -- Usuario 19: Solo abre app (bounce)
    (63, 19, 'app_open',          '2024-03-20 11:05:00', '{"source":"organic"}'),

    -- Usuario 20: Llega hasta play_song
    (64, 20, 'app_open',          '2024-03-25 14:20:00', '{"source":"ads"}'),
    (65, 20, 'search_song',       '2024-03-25 14:23:00', '{"query":"Ryan Castro"}'),
    (66, 20, 'play_song',         '2024-03-25 14:25:00', '{"song":"Muevelo","artist":"Ryan Castro"}');

-- === EVENTOS DE RETORNO (usuarios que regresan después) ===
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
    -- Usuario 1 regresa semana 1 y semana 3
    (67, 1, 'app_open',    '2024-01-10 20:00:00', '{"source":"direct"}'),
    (68, 1, 'play_song',   '2024-01-10 20:05:00', '{"song":"Yonaguni","artist":"Bad Bunny"}'),
    (69, 1, 'app_open',    '2024-01-24 18:00:00', '{"source":"notification"}'),
    (70, 1, 'play_song',   '2024-01-24 18:10:00', '{"song":"Dakiti","artist":"Bad Bunny"}'),

    -- Usuario 3 regresa semana 2
    (71, 3, 'app_open',    '2024-01-20 14:00:00', '{"source":"notification"}'),
    (72, 3, 'play_song',   '2024-01-20 14:05:00', '{"song":"Lady Gaga","artist":"Peso Pluma"}'),

    -- Usuario 6 regresa semana 1
    (73, 6, 'app_open',    '2024-01-27 10:00:00', '{"source":"organic"}'),
    (74, 6, 'play_song',   '2024-01-27 10:05:00', '{"song":"Feliz Cumpleaños","artist":"Feid"}'),

    -- Usuario 8 regresa semana 1 y semana 2
    (75, 8, 'app_open',    '2024-02-09 08:00:00', '{"source":"notification"}'),
    (76, 8, 'play_song',   '2024-02-09 08:10:00', '{"song":"Colores","artist":"J Balvin"}'),
    (77, 8, 'app_open',    '2024-02-17 19:00:00', '{"source":"direct"}'),
    (78, 8, 'play_song',   '2024-02-17 19:05:00', '{"song":"Agua","artist":"J Balvin"}'),

    -- Usuario 13 regresa semana 1
    (79, 13, 'app_open',   '2024-02-29 16:00:00', '{"source":"organic"}'),
    (80, 13, 'play_song',  '2024-02-29 16:10:00', '{"song":"Secreto","artist":"Anuel AA"}'),

    -- Usuario 15 regresa semana 1 y semana 2
    (81, 15, 'app_open',   '2024-03-08 12:00:00', '{"source":"notification"}'),
    (82, 15, 'play_song',  '2024-03-08 12:05:00', '{"song":"Memorias","artist":"Mora"}'),
    (83, 15, 'app_open',   '2024-03-16 09:00:00', '{"source":"direct"}'),

    -- Usuario 2 regresa semana 2 (tardío)
    (84, 2, 'app_open',    '2024-01-19 11:00:00', '{"source":"ads"}'),
    (85, 2, 'play_song',   '2024-01-19 11:05:00', '{"song":"Waka Waka","artist":"Shakira"}'),

    -- Usuario 10 regresa semana 1
    (86, 10, 'app_open',   '2024-02-17 10:00:00', '{"source":"referral"}'),
    (87, 10, 'play_song',  '2024-02-17 10:05:00', '{"song":"Dile Que Tu Me Quieres","artist":"Ozuna"}');

## 0.4 Verificación rápida

Completa primero tus predicciones:

| Validación | Mi predicción |
|---|---|
| Total de usuarios | |
| Total de eventos | |
| Evento más frecuente | |

Cuando ejecutes las consultas, registra debajo:

| Validación | Resultado real |
|---|---|
| Total de usuarios | |
| Total de eventos | |
| Evento más frecuente | |

### Observaciones
- ¿Los volúmenes te parecen razonables? ______________________________________
- ¿Hay algo que te llame la atención del journey? _______________________________

In [ ]:
-- Verificación rápida
SELECT COUNT(*) AS total_users FROM users;
SELECT COUNT(*) AS total_events FROM events;

SELECT
    event_name,
    COUNT(*) AS cantidad
FROM events
GROUP BY event_name
ORDER BY cantidad DESC;

-- Opcional:
-- agrega una validación extra que te ayude a confiar en el dataset

---
# Bloque 0 · Breakout Rooms — Calentamiento (10 min)

Trabaja con tu equipo. Antes de escribir la consulta, discutan la lógica.

### Ejercicio 0.1 — Exploración de eventos únicos (Nivel: ⭐)

**Descripción:** Lista todos los tipos de evento (`event_name`) distintos que existen en la tabla `events`, ordenados alfabéticamente.

**Objetivo:** Familiarizarte con el dataset y verificar que las etapas del journey estén presentes.

**Piensa antes de escribir:**

| Pregunta | Tu respuesta |
|---|---|
| ¿Qué columna necesitas revisar? | |
| ¿Qué palabra clave elimina duplicados? | |
| ¿Cómo ordenarías alfabéticamente? | |

**Pista:** Usa `DISTINCT` sobre `event_name` y ordena por la misma columna.

**Plantilla base:**

```sql
SELECT DISTINCT ______________________
FROM ______________________
ORDER BY ______________________;
```

**Interpretación:**

- ¿Cuántos eventos distintos encontraste?

- ¿Coinciden con el journey esperado de MusicFlow?

- ¿Falta algún evento que te hubiera gustado medir?


In [ ]:
-- Ejercicio 0.1
-- Escribe aquí tu consulta.


### Ejercicio 0.2 — Usuarios activos por mes (Nivel: ⭐)

**Descripción:** Cuenta cuántos usuarios únicos abrieron la app (`app_open`) en cada mes del Q1 2024.

**Objetivo:** Practicar `COUNT(DISTINCT ...)` con agrupación temporal.

**Piensa antes de escribir:**

| Elemento | Tu respuesta |
|---|---|
| Evento a filtrar | |
| Función para truncar al mes | |
| Campo a contar sin duplicar | |

**Pista:** Combina `DATE_TRUNC('month', event_ts)` con `COUNT(DISTINCT user_id)`.

**Plantilla base:**

```sql
SELECT
    ______________________________ AS mes,
    COUNT(DISTINCT ______________________________) AS usuarios_unicos
FROM ______________________________
WHERE ______________________________
GROUP BY ______________________________
ORDER BY ______________________________;
```

**Interpretación:**

- ¿Qué mes tuvo más usuarios activos?

- ¿Notas alguna tendencia entre enero, febrero y marzo?

- ¿Este resultado habla de adquisición o de actividad?


In [ ]:
-- Ejercicio 0.2
-- Escribe aquí tu consulta.


### Ejercicio 0.3 — Usuarios con playlist pero sin suscripción (Nivel: ⭐⭐)

**Descripción:** Identifica los usuarios que crearon una playlist pero nunca se suscribieron a premium.

**Objetivo:** Usar CTEs y un `LEFT JOIN` para detectar journeys incompletos.

**Piensa antes de escribir:**

| Paso lógico | ¿Qué haré? |
|---|---|
| CTE 1 | |
| CTE 2 | |
| Tipo de join | |
| Condición final para detectar ausentes | |

**Pista:** Piensa en un set difference: usuarios de playlist menos usuarios de premium.

**Plantilla base:**

```sql
WITH playlist_cte AS (
    SELECT DISTINCT ______________________
    FROM ______________________
    WHERE ______________________
),
premium_cte AS (
    SELECT DISTINCT ______________________
    FROM ______________________
    WHERE ______________________
)
SELECT
    p.____________________
FROM playlist_cte p
LEFT JOIN premium_cte s
    ON p.____________________ = s.____________________
WHERE s.____________________ IS NULL
ORDER BY 1;
```

**Interpretación:**

- ¿Cuántos usuarios llegaron a crear playlist pero no convirtieron?

- ¿Qué acción podría proponer producto para este grupo?

- ¿Este grupo parece cercano a convertir o no necesariamente?


In [ ]:
-- Ejercicio 0.3
-- Escribe aquí tu consulta.


---
# Bloque 1 · Coding Together — CTEs aplicadas (25 min)

En este bloque vamos a practicar cómo dividir problemas complejos en pasos legibles.

### Ejercicio 1.1 — Resumen de actividad por usuario con CTE (Nivel: ⭐)

**Descripción:** Crea una CTE llamada `actividad` que calcule para cada usuario su total de eventos, primer evento y último evento. Luego muestra solo los 5 usuarios más activos.

**Objetivo:** Practicar la sintaxis básica de una CTE con agregaciones.

**Piensa antes de escribir:**

| Campo a calcular | Función agregada |
|---|---|
| total de eventos | |
| primer timestamp | |
| último timestamp | |

**Pista:** Usa `COUNT(*)`, `MIN(event_ts)` y `MAX(event_ts)`.

**Plantilla base:**

```sql
WITH actividad AS (
    SELECT
        ______________________ AS user_id,
        ______________________ AS total_eventos,
        ______________________ AS primer_evento_ts,
        ______________________ AS ultimo_evento_ts
    FROM ______________________
    GROUP BY ______________________
)
SELECT *
FROM actividad
ORDER BY ______________________ DESC
LIMIT 5;
```

**Interpretación:**

- ¿Quiénes aparecen como más activos?

- ¿Ser más activo implica necesariamente ser premium?

- ¿Qué otra métrica agregarías a esta CTE?


In [ ]:
-- Ejercicio 1.1
-- Escribe aquí tu consulta.


### Ejercicio 1.2 — Dos CTEs con JOIN: suscriptores y atributos (Nivel: ⭐)

**Descripción:** Crea una CTE `suscriptores` con los usuarios que hicieron `subscribe_premium` y otra CTE `info_user` con país, canal y device. Luego crúzalas.

**Objetivo:** Practicar múltiples CTEs y enriquecer datos de eventos con datos maestros.

**Piensa antes de escribir:**

| CTE | ¿Qué contendrá? |
|---|---|
| `suscriptores` | |
| `info_user` | |

**Pista:** La primera CTE sale de `events`; la segunda sale de `users`.

**Plantilla base:**

```sql
WITH suscriptores AS (
    SELECT DISTINCT ______________________
    FROM ______________________
    WHERE ______________________
),
info_user AS (
    SELECT
        ______________________,
        ______________________,
        ______________________,
        ______________________
    FROM ______________________
)
SELECT
    s.____________________,
    i.____________________,
    i.____________________,
    i.____________________
FROM suscriptores s
INNER JOIN info_user i
    ON s.____________________ = i.____________________
ORDER BY 1;
```

**Interpretación:**

- ¿Qué países aparecen entre los premium?

- ¿Ves algún canal repetirse más entre quienes convierten?

- ¿Qué análisis adicional harías con este resultado?


In [ ]:
-- Ejercicio 1.2
-- Escribe aquí tu consulta.


### Ejercicio 1.3 — CTEs encadenadas: pipeline de transformaciones (Nivel: ⭐⭐)

**Descripción:** Construye 3 CTEs encadenadas: eventos de febrero, conteo por usuario y segmento de actividad.

**Objetivo:** Entender cómo una CTE puede consumir la anterior y formar un pipeline.

**Piensa antes de escribir:**

| Paso | Resultado esperado |
|---|---|
| `eventos_febrero` | |
| `conteo_por_usuario` | |
| `segmento` | |

**Pista:** Cada CTE debe partir de la anterior, no de la tabla original.

**Plantilla base:**

```sql
WITH eventos_febrero AS (
    SELECT *
    FROM events
    WHERE event_ts >= '2024-02-01'
      AND event_ts <  '2024-03-01'
),
conteo_por_usuario AS (
    SELECT
        ______________________,
        ______________________ AS total_eventos
    FROM eventos_febrero
    GROUP BY ______________________
),
segmento AS (
    SELECT
        ______________________,
        total_eventos,
        CASE
            WHEN total_eventos >= 5 THEN 'Alta actividad'
            WHEN total_eventos >= 3 THEN 'Media'
            ELSE 'Baja'
        END AS segmento_actividad
    FROM conteo_por_usuario
)
SELECT
    ______________________,
    COUNT(*) AS n_usuarios
FROM segmento
GROUP BY ______________________
ORDER BY n_usuarios DESC;
```

**Interpretación:**

- ¿Qué segmento domina en febrero?

- ¿El número de eventos parece concentrado en pocos usuarios o repartido?

- ¿Cómo usarías esta segmentación en CRM o marketing?


In [ ]:
-- Ejercicio 1.3
-- Escribe aquí tu consulta.


### Ejercicio 1.4 — Primer evento de cada usuario con `ROW_NUMBER()` (Nivel: ⭐⭐⭐)

**Descripción:** Usa una CTE con `ROW_NUMBER()` para identificar el primer evento de cada usuario y luego crúzalo con `users`.

**Objetivo:** Combinar CTEs con funciones de ventana para reconstruir journeys.

**Piensa antes de escribir:**

| Elemento | Tu respuesta |
|---|---|
| Columna para particionar | |
| Columna para ordenar | |
| Valor de `rn` que me interesa | |

**Pista:** Usa `ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_ts)`.

**Plantilla base:**

```sql
WITH primer_evento AS (
    SELECT
        user_id,
        event_name,
        event_ts,
        ROW_NUMBER() OVER (
            PARTITION BY ______________________
            ORDER BY ______________________
        ) AS rn
    FROM events
)
SELECT
    u.user_id,
    u.country,
    u.channel,
    p.event_name AS primer_evento,
    p.event_ts
FROM users u
INNER JOIN primer_evento p
    ON u.user_id = p.user_id
WHERE p.rn = ______________________
ORDER BY u.user_id;
```

**Interpretación:**

- ¿Todos los usuarios empiezan igual su journey?

- ¿Qué pasaría si existieran eventos de onboarding antes de `app_open`?

- ¿En qué casos esta técnica es útil en negocio?


In [ ]:
-- Ejercicio 1.4
-- Escribe aquí tu consulta.


---
# Bloque 2 · Coding Together — Funnel de conversión (25 min)

Journey del funnel:

`app_open → search_song → play_song → create_playlist → subscribe_premium`

### Ejercicio 2.1 — Funnel completo del Q1 2024 (Nivel: ⭐⭐)

**Descripción:** Construye el funnel completo y calcula usuarios por etapa, drop-off entre etapas consecutivas y tasa de conversión global.

**Objetivo:** Traducir el journey del producto a una estructura analítica con CTEs.

**Piensa antes de escribir:**

| Etapa | Nombre de la CTE |
|---|---|
| app_open | |
| search_song | |
| play_song | |
| create_playlist | |
| subscribe_premium | |

**Pista:** El drop-off es la resta entre etapas. La conversión global compara el final contra el inicio.

**Plantilla base:**

```sql
WITH
f_open AS (
    SELECT DISTINCT user_id
    FROM events
    WHERE event_name = 'app_open'
),
f_search AS (
    SELECT DISTINCT user_id
    FROM events
    WHERE event_name = 'search_song'
),
f_play AS (
    SELECT DISTINCT user_id
    FROM events
    WHERE event_name = 'play_song'
),
f_playlist AS (
    SELECT DISTINCT user_id
    FROM events
    WHERE event_name = 'create_playlist'
),
f_premium AS (
    SELECT DISTINCT user_id
    FROM events
    WHERE event_name = 'subscribe_premium'
),
counts AS (
    SELECT
        (SELECT COUNT(*) FROM f_open)      AS open_users,
        (SELECT COUNT(*) FROM f_search)    AS search_users,
        (SELECT COUNT(*) FROM f_play)      AS play_users,
        (SELECT COUNT(*) FROM f_playlist)  AS playlist_users,
        (SELECT COUNT(*) FROM f_premium)   AS premium_users
)
SELECT
    *,
    ____________________________________________ AS drop_open_to_search,
    ____________________________________________ AS drop_search_to_play,
    ____________________________________________ AS drop_play_to_playlist,
    ____________________________________________ AS drop_playlist_to_premium,
    ____________________________________________ AS conversion_global_pct
FROM counts;
```

**Interpretación:**

- ¿Cuál fue la mayor caída del funnel?

- ¿Qué porcentaje de quienes abren termina convirtiendo a premium?

- ¿Qué acción de producto propondrías con base en este funnel?


In [ ]:
-- Ejercicio 2.1
-- Escribe aquí tu consulta.


### Ejercicio 2.2 — Funnel solo de febrero 2024 (Nivel: ⭐⭐)

**Descripción:** Repite el funnel, pero filtrando solamente eventos de febrero 2024.

**Objetivo:** Practicar filtros temporales dentro de una CTE base.

**Piensa antes de escribir:**

| Pregunta | Tu respuesta |
|---|---|
| Fecha inicial | |
| Fecha final exclusiva | |
| ¿Qué CTE debería usar el resto de etapas? | |

**Pista:** Crea una CTE `base` y reutilízala en todas las etapas.

**Plantilla base:**

```sql
WITH base AS (
    SELECT *
    FROM events
    WHERE event_ts >= '2024-02-01'
      AND event_ts <  '2024-03-01'
),
f_open AS (
    SELECT DISTINCT user_id FROM base WHERE event_name = 'app_open'
),
f_search AS (
    SELECT DISTINCT user_id FROM base WHERE event_name = 'search_song'
),
f_play AS (
    SELECT DISTINCT user_id FROM base WHERE event_name = 'play_song'
),
f_playlist AS (
    SELECT DISTINCT user_id FROM base WHERE event_name = 'create_playlist'
),
f_premium AS (
    SELECT DISTINCT user_id FROM base WHERE event_name = 'subscribe_premium'
)
SELECT
    ____________________________________________ AS open_users,
    ____________________________________________ AS search_users,
    ____________________________________________ AS play_users,
    ____________________________________________ AS playlist_users,
    ____________________________________________ AS premium_users;
```

**Interpretación:**

- ¿Qué cambió frente al funnel general?

- ¿Febrero parece mejor, peor o similar?

- ¿Qué conclusión sí puedes sacar y cuál no deberías sobregeneralizar?


In [ ]:
-- Ejercicio 2.2
-- Escribe aquí tu consulta.


### Ejercicio 2.3 — Tiempos entre etapas del journey (Nivel: ⭐⭐⭐)

**Descripción:** Para cada usuario, calcula el tiempo en minutos entre etapas consecutivas del journey usando el primer timestamp de cada evento.

**Objetivo:** Medir fricción y latencia dentro del proceso de conversión.

**Piensa antes de escribir:**

| Etapa | ¿Qué necesitas guardar? |
|---|---|
| open | |
| search | |
| play | |
| playlist | |
| premium | |

**Pista:** Usa `DATEDIFF('minute', ts_anterior, ts_actual)`.

**Plantilla base:**

```sql
WITH
o  AS (
    SELECT user_id, MIN(event_ts) AS open_ts
    FROM events
    WHERE event_name = 'app_open'
    GROUP BY user_id
),
s  AS (
    SELECT user_id, MIN(event_ts) AS search_ts
    FROM events
    WHERE event_name = 'search_song'
    GROUP BY user_id
),
p  AS (
    SELECT user_id, MIN(event_ts) AS play_ts
    FROM events
    WHERE event_name = 'play_song'
    GROUP BY user_id
),
pl AS (
    SELECT user_id, MIN(event_ts) AS playlist_ts
    FROM events
    WHERE event_name = 'create_playlist'
    GROUP BY user_id
),
pr AS (
    SELECT user_id, MIN(event_ts) AS premium_ts
    FROM events
    WHERE event_name = 'subscribe_premium'
    GROUP BY user_id
)
SELECT
    o.user_id,
    ____________________________________________ AS min_open_to_search,
    ____________________________________________ AS min_search_to_play,
    ____________________________________________ AS min_play_to_playlist,
    ____________________________________________ AS min_playlist_to_premium
FROM o
LEFT JOIN s  ON o.user_id = s.user_id
LEFT JOIN p  ON o.user_id = p.user_id
LEFT JOIN pl ON o.user_id = pl.user_id
LEFT JOIN pr ON o.user_id = pr.user_id
ORDER BY o.user_id;
```

**Interpretación:**

- ¿En qué transición los usuarios parecen demorarse más?

- ¿Qué significa un `NULL` en este contexto?

- ¿Qué hipótesis de UX podrías formular a partir de estos tiempos?


In [ ]:
-- Ejercicio 2.3
-- Escribe aquí tu consulta.


### 2.4 Discusión — Interpretación del funnel

Completa con tu equipo:

| Pregunta | Respuesta |
|---|---|
| Mayor cuello de botella | |
| Hipótesis de negocio | |
| Acción recomendada para producto | |

### Mi resumen ejecutivo
Escribe una recomendación en 3–4 líneas como si se la fueras a contar a un PM:

_______________________________________________________________

_______________________________________________________________

_______________________________________________________________

---
# Bloque 3 · Coding Together — Cohorts y retención (20 min)

### Ejercicio 3.1 — Identificar cohorts mensuales (Nivel: ⭐)

**Descripción:** Muestra las cohortes mensuales y, para cada una, cuántos usuarios hay y cuántos eventualmente se suscribieron a premium.

**Objetivo:** Asignar cohorts y cruzarlas con una métrica simple de conversión.

**Piensa antes de escribir:**

| Elemento | Tu respuesta |
|---|---|
| Campo de signup | |
| Función para cohort mensual | |
| Evento que indica conversión | |

**Pista:** La cohort se deriva de `signup_ts`, no de `event_ts`.

**Plantilla base:**

```sql
WITH premium AS (
    SELECT DISTINCT user_id
    FROM events
    WHERE event_name = 'subscribe_premium'
)
SELECT
    ______________________________ AS cohort_month,
    COUNT(*) AS cohort_size,
    COUNT(p.user_id) AS premium_users
FROM users u
LEFT JOIN premium p
    ON u.user_id = p.user_id
GROUP BY ______________________________
ORDER BY ______________________________;
```

**Interpretación:**

- ¿Qué cohort tiene más usuarios?

- ¿Qué cohort parece convertir mejor en términos absolutos?

- ¿Qué métrica relativa faltaría para comparar mejor?


In [ ]:
-- Ejercicio 3.1
-- Escribe aquí tu consulta.


### Ejercicio 3.2 — Tabla de retención semanal (Nivel: ⭐⭐⭐)

**Descripción:** Construye una tabla de retención por cohort mensual y semanas 0, 1, 2 y 3.

**Objetivo:** Entender la lógica de retención por ventanas temporales.

**Piensa antes de escribir:**

| Paso | ¿Qué hará tu CTE? |
|---|---|
| `u` | |
| `grid` | |
| `retention` | |

**Pista:** El porcentaje se calcula como retained_users / cohort_size * 100.

**Plantilla base:**

```sql
WITH
u AS (
    SELECT
        user_id,
        DATE_TRUNC('month', signup_ts) AS cohort_month
    FROM users
),
grid AS (
    SELECT UNNEST(ARRAY[0, 1, 2, 3]) AS week_offset
),
retention AS (
    SELECT
        u.cohort_month,
        g.week_offset,
        COUNT(*) AS cohort_size,
        SUM(
            CASE
                WHEN EXISTS (
                    SELECT 1
                    FROM events e
                    WHERE e.user_id = u.user_id
                      AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) >= 7 * g.week_offset
                      AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) <  7 * (g.week_offset + 1)
                )
                THEN 1 ELSE 0
            END
        ) AS retained_users
    FROM u
    CROSS JOIN grid g
    GROUP BY u.cohort_month, g.week_offset
)
SELECT
    cohort_month,
    week_offset,
    cohort_size,
    retained_users,
    ____________________________________________ AS retention_pct
FROM retention
ORDER BY cohort_month, week_offset;
```

**Interpretación:**

- ¿Qué significa exactamente `week_offset = 0`?

- ¿Cuál cohort parece retener mejor?

- ¿La retención cae rápido o lentamente?


In [ ]:
-- Ejercicio 3.2
-- Escribe aquí tu consulta.


### Ejercicio 3.3 — Tabla pivote para heatmap (Nivel: ⭐⭐⭐)

**Descripción:** Convierte la tabla anterior a formato ancho: una fila por cohort y una columna por semana.

**Objetivo:** Preparar el resultado para visualización en Google Sheets o Excel.

**Piensa antes de escribir:**

| Columna | ¿Qué representa? |
|---|---|
| `w0` | |
| `w1` | |
| `w2` | |
| `w3` | |

**Pista:** Haz un pivote manual con `MAX(CASE WHEN ...)`.

**Plantilla base:**

```sql
WITH pct AS (
    -- Reutiliza aquí tu lógica del ejercicio 3.2
    SELECT
        cohort_month,
        week_offset,
        retention_pct
    FROM ______________________________
)
SELECT
    cohort_month,
    MAX(CASE WHEN week_offset = 0 THEN retention_pct END) AS w0,
    MAX(CASE WHEN week_offset = 1 THEN retention_pct END) AS w1,
    MAX(CASE WHEN week_offset = 2 THEN retention_pct END) AS w2,
    MAX(CASE WHEN week_offset = 3 THEN retention_pct END) AS w3
FROM pct
GROUP BY cohort_month
ORDER BY cohort_month;
```

**Interpretación:**

- ¿Qué fila debería verse más intensa en un heatmap?

- ¿Qué ventaja tiene llevar esto a una herramienta visual?

- ¿Qué historia le contarías a negocio con esta tabla?


In [ ]:
-- Ejercicio 3.3
-- Escribe aquí tu consulta.


### Ejercicio 3.4 — Retención segmentada por canal (Nivel: ⭐⭐⭐)

**Descripción:** Extiende la tabla de retención para incluir el canal de adquisición.

**Objetivo:** Comparar curvas de retención entre segmentos de negocio.

**Piensa antes de escribir:**

| Segmento a agregar | ¿De qué tabla sale? |
|---|---|
| `channel` | |

**Pista:** Agrega `channel` desde el principio y propágalo por los `GROUP BY`.

**Plantilla base:**

```sql
WITH
u AS (
    SELECT
        user_id,
        DATE_TRUNC('month', signup_ts) AS cohort_month,
        ______________________________ AS channel
    FROM users
),
grid AS (
    SELECT UNNEST(ARRAY[0, 1, 2, 3]) AS week_offset
)
SELECT
    u.cohort_month,
    u.channel,
    g.week_offset,
    COUNT(*) AS cohort_size,
    SUM(
        CASE
            WHEN EXISTS (
                SELECT 1
                FROM events e
                WHERE e.user_id = u.user_id
                  AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) >= 7 * g.week_offset
                  AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) <  7 * (g.week_offset + 1)
            )
            THEN 1 ELSE 0
        END
    ) AS retained_users
FROM u
CROSS JOIN grid g
GROUP BY
    u.cohort_month,
    u.channel,
    g.week_offset
ORDER BY
    u.cohort_month,
    u.channel,
    g.week_offset;
```

**Interpretación:**

- ¿Qué canal parece retener mejor?

- ¿Convierte mejor el mismo canal que retiene mejor?

- ¿Qué decisión de adquisición podrías revisar con esta evidencia?


In [ ]:
-- Ejercicio 3.4
-- Escribe aquí tu consulta.


---
# Bloque 4 · Breakout Rooms — Ejercicio integrador (15 min)

En este bloque aplica lo aprendido y deja evidencia de tu razonamiento.

### Ejercicio 4.1 — Journey individual con tiempos (Nivel: ⭐⭐)

**Descripción:** Reconstruye el journey completo del usuario 8, ordenando sus eventos y calculando el tiempo entre cada evento y el anterior.

**Objetivo:** Usar `LAG()` para analizar secuencias de eventos.

**Piensa antes de escribir:**

| Elemento | Tu respuesta |
|---|---|
| Usuario a analizar | |
| Función para ver el evento anterior | |
| Campo para ordenar cronológicamente | |

**Pista:** `LAG()` te da el timestamp anterior dentro del mismo usuario.

**Plantilla base:**

```sql
WITH journey AS (
    SELECT
        user_id,
        event_name,
        event_ts,
        LAG(event_ts) OVER (
            PARTITION BY ______________________
            ORDER BY ______________________
        ) AS prev_ts
    FROM events
    WHERE user_id = ______________________
)
SELECT
    user_id,
    event_name,
    event_ts,
    prev_ts,
    ____________________________________________ AS min_desde_evento_previo
FROM journey
ORDER BY event_ts;
```

**Interpretación:**

- ¿Cómo fue el journey del usuario 8?

- ¿En qué parte se demoró más?

- ¿Parece un usuario de alta intención?


In [ ]:
-- Ejercicio 4.1
-- Escribe aquí tu consulta.


### Ejercicio 4.2 — Funnel por país (Nivel: ⭐⭐)

**Descripción:** Construye un funnel resumido `app_open → subscribe_premium` agrupado por país.

**Objetivo:** Segmentar el funnel por un atributo de usuario.

**Piensa antes de escribir:**

| Pregunta | Tu respuesta |
|---|---|
| ¿Qué tablas necesitas unir? | |
| ¿Qué atributo segmenta el resultado? | |
| ¿Qué métrica requiere `DISTINCT`? | |

**Pista:** La tasa compara premium_users contra open_users.

**Plantilla base:**

```sql
WITH base AS (
    SELECT
        e.user_id,
        e.event_name,
        u.country
    FROM events e
    INNER JOIN users u
        ON e.user_id = u.user_id
)
SELECT
    country,
    COUNT(DISTINCT CASE WHEN event_name = 'app_open' THEN user_id END) AS open_users,
    COUNT(DISTINCT CASE WHEN event_name = 'subscribe_premium' THEN user_id END) AS premium_users,
    ____________________________________________ AS conversion_pct
FROM base
GROUP BY country
ORDER BY conversion_pct DESC;
```

**Interpretación:**

- ¿Qué país convierte mejor?

- ¿Ese resultado puede estar afectado por tamaño de muestra?

- ¿Qué otra segmentación harías después de país?


In [ ]:
-- Ejercicio 4.2
-- Escribe aquí tu consulta.


### Ejercicio 4.3 — Clasificación de usuarios por avance en el journey (Nivel: ⭐⭐⭐)

**Descripción:** Clasifica a cada usuario según la etapa máxima alcanzada: Bounce, Explorador, Oyente, Creador o Premium.

**Objetivo:** Combinar flags por usuario con una clasificación condicional.

**Piensa antes de escribir:**

| Flag | ¿Qué evento lo activa? |
|---|---|
| `has_open` | |
| `has_search` | |
| `has_play` | |
| `has_playlist` | |
| `has_premium` | |

**Pista:** El porcentaje puede calcularse contra el total de usuarios clasificados.

**Plantilla base:**

```sql
WITH etapas AS (
    SELECT
        user_id,
        MAX(CASE WHEN event_name = 'app_open' THEN 1 ELSE 0 END) AS has_open,
        MAX(CASE WHEN event_name = 'search_song' THEN 1 ELSE 0 END) AS has_search,
        MAX(CASE WHEN event_name = 'play_song' THEN 1 ELSE 0 END) AS has_play,
        MAX(CASE WHEN event_name = 'create_playlist' THEN 1 ELSE 0 END) AS has_playlist,
        MAX(CASE WHEN event_name = 'subscribe_premium' THEN 1 ELSE 0 END) AS has_premium
    FROM events
    GROUP BY user_id
),
clasificacion AS (
    SELECT
        user_id,
        CASE
            WHEN has_premium = 1 THEN 'Premium'
            WHEN has_playlist = 1 THEN 'Creador'
            WHEN has_play = 1 THEN 'Oyente'
            WHEN has_search = 1 THEN 'Explorador'
            ELSE 'Bounce'
        END AS etapa_maxima
    FROM etapas
)
SELECT
    etapa_maxima,
    COUNT(*) AS n_users,
    ____________________________________________ AS pct_users
FROM clasificacion
GROUP BY etapa_maxima
ORDER BY n_users DESC;
```

**Interpretación:**

- ¿Qué categoría concentra más usuarios?

- ¿Qué te dice esto sobre el journey real?

- ¿Cómo usarías esta clasificación en una campaña o dashboard?


In [ ]:
-- Ejercicio 4.3
-- Escribe aquí tu consulta.


### Ejercicio 4.4 — Checks de calidad de datos (Nivel: ⭐⭐)

**Descripción:** Valida duplicados exactos, eventos en el futuro y usuarios que se suscribieron sin haber reproducido canciones.

**Objetivo:** Recordar que antes de analizar hay que confiar en la calidad del dato.

**Piensa antes de escribir:**

Completa esta mini bitácora:

| Check | ¿Qué espero encontrar? | Resultado | ¿Es un problema? |
|---|---|---|---|
| Duplicados exactos | | | |
| Eventos en el futuro | | | |
| Premium sin `play_song` | | | |

**Pista:** Documenta no solo si hay errores, sino qué impacto tendrían.

**Plantilla base:**

```sql
-- Check 1: duplicados exactos
SELECT
    user_id,
    event_name,
    event_ts,
    COUNT(*) AS veces
FROM events
GROUP BY user_id, event_name, event_ts
HAVING COUNT(*) > 1;

-- Check 2: eventos en el futuro
SELECT *
FROM events
WHERE event_ts > CURRENT_TIMESTAMP;

-- Check 3: premium sin play_song
WITH flags AS (
    SELECT
        user_id,
        MAX(CASE WHEN event_name = 'play_song' THEN 1 ELSE 0 END) AS has_play,
        MAX(CASE WHEN event_name = 'subscribe_premium' THEN 1 ELSE 0 END) AS has_premium
    FROM events
    GROUP BY user_id
)
SELECT *
FROM flags
WHERE has_premium = 1
  AND has_play = 0;
```

**Interpretación:**

- ¿Puedes confiar en este dataset para presentar resultados?

- ¿Qué check adicional agregarías?

- ¿Cuál de estos errores sería más grave en un proyecto real?


In [ ]:
-- Ejercicio 4.4
-- Ejecuta los checks y agrega aquí cualquier validación extra.


---
# Cierre (5 min)

## Discusión en plenaria

Completa al final:

1. **Mayor cuello de botella del funnel:**  
   ________________________________________________________________

2. **Hallazgo clave de cohorts/retención:**  
   ________________________________________________________________

3. **Hallazgo clave por segmento (canal, país, etc.):**  
   ________________________________________________________________

4. **Una acción concreta que propondrías a negocio:**  
   ________________________________________________________________

## Takeaways personales
- Lo que mejor entendí hoy fue: _____________________________________________
- Lo que todavía necesito practicar es: _____________________________________
- El ejercicio que más me costó fue: ________________________________________

### ¿Necesitas ayuda?
Recuerda los canales de ayuda que tenemos disponibles:
- `DATA-CO-LEARNING`: Revisa los horarios de atencion en la guia dele studiante. Recuerda que tenemos horarios de apoyo todos los días para tus dudas puntuales!
- `DA_CONSULTA`: Uuedes publicar tus preguntas sobre el contenido de la plataforma o el proyecto 24/7. Recuerda que en tu pregunta debes de agregar el tag @Dataconsulta.
- `SPRINT FOCUS`: Para los Sprints 1 al 5 tenemos sesiones extra donde abordamos temáticas de cada sprint a rpofundidad. Revisa la guia del estudiante para conocer los horarios!
- `SESIONES 1:1`: Recuerda que puedes agendar sesiones 1:1 con un tutor. Agendalas con antelación y resuelve todas tus dudas, desde temás del proyecto, curso, carrera hasta problemas técnicos.
- `CANAL DE DISCORD DE COMMUNITY`: Recuerda que tu cohorte tiene un canal especial donde puedes compartir cualquier item interesante que quieras mostrar a tus compañeros de curso.

### Mis dudas pendientes
- ________________________________________________________________
- ________________________________________________________________
- ________________________________________________________________

## Siguientes pasos

- **Próxima sesión:** profundización en experimentación, funnels o análisis relacionado con producto.
- **Tarea sugerida:** exporta la tabla pivote de retención a Google Sheets y crea un heatmap.
- **Desafío extra:** agrega una sexta etapa al funnel, por ejemplo `share_playlist`, y evalúa cómo cambia la conversión.

## Resumen de ejercicios

| # | Ejercicio | Nivel | Tema |
|---|---|---|---|
| 0.1 | Eventos únicos | ⭐ | Exploración |
| 0.2 | Usuarios activos por mes | ⭐ | Exploración temporal |
| 0.3 | Playlist sin suscripción | ⭐⭐ | Set difference + CTE |
| 1.1 | Resumen de actividad | ⭐ | CTE básica |
| 1.2 | Suscriptores y atributos | ⭐ | CTE + JOIN |
| 1.3 | Pipeline encadenado | ⭐⭐ | CTEs encadenadas |
| 1.4 | Primer evento | ⭐⭐⭐ | Window + CTE |
| 2.1 | Funnel completo | ⭐⭐ | Funnel |
| 2.2 | Funnel de febrero | ⭐⭐ | Funnel temporal |
| 2.3 | Tiempos entre etapas | ⭐⭐⭐ | Journey |
| 3.1 | Cohorts mensuales | ⭐ | Cohorts |
| 3.2 | Retención semanal | ⭐⭐⭐ | Retención |
| 3.3 | Tabla pivote | ⭐⭐⭐ | Visualización |
| 3.4 | Retención por canal | ⭐⭐⭐ | Segmentación |
| 4.1 | Journey del usuario 8 | ⭐⭐ | LAG |
| 4.2 | Funnel por país | ⭐⭐ | Segmentación |
| 4.3 | Clasificación de usuarios | ⭐⭐⭐ | Segmentación avanzada |
| 4.4 | Calidad de datos | ⭐⭐ | Data QA |

### Autoevaluación final
Califica del 1 al 5 cómo te sentiste hoy:

- CTEs: ___ / 5
- Funnel: ___ / 5
- Cohorts: ___ / 5
- Interpretación de negocio: ___ / 5